In [1]:
# ── Cell 0:把工作目錄切到專案根目錄 ──
# notebook 預設的 cwd 是它自己所在的資料夾(src/gesture_demo/),
# 這樣 import src.gesture_demo.* 會找不到(No module named 'src'),
# "data/raw/gestures.csv" 這種相對路徑也對不上。
# 往上找到含有 src/ 的那層當專案根,加進 sys.path 並切過去。
import os, sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "src").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("專案根目錄:", ROOT)

# ── Cell 1:import ──
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.gesture_demo.model import GestureMLP
from src.gesture_demo.dataset import GestureDataset
from src.gesture_demo.session_split import session_train_test_split

專案根目錄: c:\Users\user\Gesture_demo


In [2]:
# ── Cell 2:讀資料 + 切分 ──
cols = ["label", "session_id"] + [f"{ax}{i}" for i in range(21) for ax in ["x", "y", "z"]]
df = pd.read_csv("data/raw/gestures.csv", header=None, names=cols)
train_df, test_df = session_train_test_split(df, test_ratio=0.25, seed=42)

train_ds = GestureDataset(train_df)
test_ds = GestureDataset(test_df)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

In [3]:
# ── Cell 3:建模型 ──
model = GestureMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [7]:
# ── Cell 4:訓練 ──
for epoch in range(40):
    model.train()
    total_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch+1:2d}: loss = {total_loss/len(train_loader):.4f}")

epoch  1: loss = 0.0070
epoch  2: loss = 0.0070
epoch  3: loss = 0.0055
epoch  4: loss = 0.0093
epoch  5: loss = 0.0061
epoch  6: loss = 0.0080
epoch  7: loss = 0.0054
epoch  8: loss = 0.0063
epoch  9: loss = 0.0057
epoch 10: loss = 0.0036
epoch 11: loss = 0.0044
epoch 12: loss = 0.0064
epoch 13: loss = 0.0039
epoch 14: loss = 0.0066
epoch 15: loss = 0.0074
epoch 16: loss = 0.0037
epoch 17: loss = 0.0034
epoch 18: loss = 0.0030
epoch 19: loss = 0.0020
epoch 20: loss = 0.0066
epoch 21: loss = 0.0027
epoch 22: loss = 0.0076
epoch 23: loss = 0.0030
epoch 24: loss = 0.0036
epoch 25: loss = 0.0030
epoch 26: loss = 0.0044
epoch 27: loss = 0.0071
epoch 28: loss = 0.0023
epoch 29: loss = 0.0023
epoch 30: loss = 0.0029
epoch 31: loss = 0.0038
epoch 32: loss = 0.0039
epoch 33: loss = 0.0049
epoch 34: loss = 0.0010
epoch 35: loss = 0.0010
epoch 36: loss = 0.0022
epoch 37: loss = 0.0075
epoch 38: loss = 0.0021
epoch 39: loss = 0.0040
epoch 40: loss = 0.0018


In [8]:
# ── Cell 5:評估 ──
from sklearn.metrics import confusion_matrix, classification_report
from src.gesture_demo.dataset import GESTURE_LABELS

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        idx = model(X_batch).argmax(dim=1)
        all_preds.extend(idx.tolist())
        all_labels.extend(y_batch.tolist())

acc = sum(p==l for p,l in zip(all_preds, all_labels)) / len(all_labels)
print(f"test accuracy: {acc:.4f}")
# labels 明確給滿:某一類還沒資料 / 沒被切進 test 時也不會報錯
all_idx = list(range(len(GESTURE_LABELS)))
print(classification_report(all_labels, all_preds,
                            labels=all_idx, target_names=GESTURE_LABELS,
                            zero_division=0))


test accuracy: 0.9243
              precision    recall  f1-score   support

        fist       0.97      0.97      0.97       532
        open       1.00      1.00      1.00       545
       point       1.00      0.96      0.98       536
        yeah       0.73      0.97      0.83       538
    thumb_up       0.91      0.94      0.93       635
       three       0.89      0.71      0.79       510
       phone       0.94      1.00      0.97       509
          ok       0.97      1.00      0.98       509
        four       0.96      1.00      0.98       573
       seven       1.00      0.99      0.99       670
       eight       0.80      0.98      0.88       584
         gun       0.92      0.60      0.73       552
       split       0.95      1.00      0.97       447
        rock       1.00      0.96      0.98       589
      middle       0.91      0.77      0.83       525

    accuracy                           0.92      8254
   macro avg       0.93      0.92      0.92      8254
weig

In [9]:
# ── Cell 6:存模型(想存才跑)──
import os
os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/gesture_mlp.pth")
print("已存")

已存
